# Modelo 3: Resultado del Partido — Regresión Logística Multinomial V2
**Machine Learning I — Universidad Externado de Colombia**

---

**Objetivo:** Construir un clasificador multinomial capaz de predecir el resultado de los partidos de la Premier League (`H` Victoria Local, `D` Empate, `A` Victoria Visitante) y superar el **baseline exigido del 49.80%** — la tasa de acierto de las propias casas de apuestas (Bet365: 54.24% en este test set).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, chi2_contingency, kruskal
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import warnings
warnings.filterwarnings('ignore')

# Identidad Visual Premier League
PL_PURPLE = '#38003c'
PL_CYAN   = '#00ff85'
PL_PINK   = '#e90052'
PL_WHITE  = '#ffffff'
sns.set_theme(style='darkgrid')


---
## PUNTO DE PARTIDA: ¿Con qué datos llegamos?

Inventario de columnas disponibles en `matches.csv`. La primera tarea es separar las variables disponibles **antes del partido** de las que solo existen **después** del pitido final — cualquier variable del segundo grupo constituye **Data Leakage**.


In [ ]:
# Carga de los tres datasets
matches  = pd.read_csv('../../data/matches.csv')
players  = pd.read_csv('../../data/players.csv', skipinitialspace=True)
players.columns = players.columns.str.strip()
xg       = pd.read_csv('../../data/xg_train.csv')
events   = pd.read_csv('../../data/events.csv')

matches['date_dt'] = pd.to_datetime(matches['date'], dayfirst=True, errors='coerce')
matches = matches.dropna(subset=['date_dt']).sort_values('date_dt').reset_index(drop=True)

print('='*70)
print('INVENTARIO DE DATASETS')
print('='*70)
print(f'  matches.csv  → {len(matches):>6,} partidos  | {matches.shape[1]} columnas')
print(f'  players.csv  → {len(players):>6,} jugadores | {players.shape[1]} columnas')
print(f'  xg_train.csv → {len(xg):>6,} disparos  | {xg.shape[1]} columnas')
print(f'  events.csv   → {len(events):>6,} eventos   | {events.shape[1]} columnas')
print()

vc = matches['ftr'].value_counts()
n  = len(matches)
print('='*70)
print('DISTRIBUCIÓN DEL TARGET — ftr (Full Time Result)')
print('='*70)
print(f'  H  (Victoria Local)    : {vc["H"]:>3} partidos  ({vc["H"]/n*100:.1f}%)')
print(f'  D  (Empate)            : {vc["D"]:>3} partidos  ({vc["D"]/n*100:.1f}%)')
print(f'  A  (Victoria Visitante): {vc["A"]:>3} partidos  ({vc["A"]/n*100:.1f}%)')
print()
print('='*70)
print('COLUMNAS DISPONIBLES Y CLASIFICACIÓN')
print('='*70)
sep = '  ' + '─'*66
print(sep)
print(f"  {'Variable':<30} {'Disponible antes del partido':^30}")
print(sep)
rows = [
    ('date / home_team / away_team', '✓  Metadatos administrativos'),
    ('referee',                      '✓  Árbitro asignado (semanas antes)'),
    ('b365h / b365d / b365a',        '✓  Cuotas Bet365 (días antes)'),
    ('avgh / avgd / avga',           '✓  Cuotas promedio multi-casas'),
    ('ftr',                          '—  TARGET — resultado final'),
    ('fthg / ftag',                  '✗  LEAKAGE — goles del partido'),
    ('hthg / htag / htr',           '✗  LEAKAGE — resultado medio tiempo'),
    ('hs / as_ / hst / ast',        '✗  LEAKAGE — tiros totales/al arco'),
    ('hf / af / hc / ac',           '✗  LEAKAGE — faltas y corners'),
    ('hy / ay / hr / ar',           '✗  LEAKAGE — tarjetas'),
]
for var, estado in rows:
    print(f'  {var:<30} {estado}')
print(sep)


---
## SECCIÓN 2: Tabla de Candidatos

El criterio de exclusión es triple: (1) **Data Leakage**, (2) **VIF** > 10, (3) **Spearman** p ≥ 0.15. Solo pasan las variables con señal estadística real y sin redundancia.


In [ ]:
print('='*82)
print('TABLA DE CANDIDATOS — DECISIÓN DE INCLUSIÓN/EXCLUSIÓN')
print('='*82)
sep = '  ' + '─'*78
print(sep)
print(f"  {'Variable':<28} {'Estado':^8}  {'Razón'}")
print(sep)
rows = [
    ('b365h',           '  ✓  ', 'Cuota victoria local — señal directa, p<0.001'),
    ('b365a',           '  ✓  ', 'Cuota victoria visitante — señal directa, p<0.001'),
    ('home_pts_last3',  '  ✓  ', 'Momentum local últimos 3 partidos — supera VIF y Spearman'),
    ('away_gf_last3',   '  ✓  ', 'Ataque visitante reciente — supera VIF y Spearman'),
    ('avgh / avga',     '  ✗  ', 'r>0.99 con b365h/b365a → colinealidad total (VIF=∞)'),
    ('b365d',           '  ✗  ', 'Spearman p=0.63 — sin señal sobre Draw (V1 confirmado)'),
    ('avgd',            '  ✗  ', 'Eliminada por VIF iterativo (VIF=143.28)'),
    ('home_gf_last3',   '  ✗  ', 'Eliminada por VIF (VIF=12.88 — solapa con home_pts)'),
    ('away_pts_last3',  '  ✗  ', 'Eliminada por VIF (VIF=16.14 — solapa con away_gf)'),
    ('home_ga_last3',   '  ✗  ', 'Spearman p=0.33 — sin asociación con resultado'),
    ('away_ga_last3',   '  ✗  ', 'Spearman p=0.21 — la defensa visitante no discrimina'),
    ('referee',         '  ✗  ', 'Kruskal-Wallis p>0.05 — sin efecto significativo'),
    ('elo_diff',        '  ✗  ', 'Laboratorio: añadir ELO reduce test acc (colineal con cuotas)'),
    ('estadísticas en vivo', '  ✗  ', 'LEAKAGE — faltas, tiros, corners, tarjetas'),
]
for var, estado, razon in rows:
    print(f'  {var:<28} {estado}  {razon}')
print(sep)


---
## SECCIÓN 3: Feature Engineering Anti-Leakage

Protocolo anti-leakage: se itera el dataset en **orden cronológico**. Para cada partido N, se leen los promedios históricos de los **partidos 1 a N-1** y solo **después** se actualizan los históricos con el resultado del partido N.


In [ ]:
PRIOR_PTS   = 1.30   # media histórica puntos/partido PL
PRIOR_GOALS = 1.38   # media histórica goles/partido PL
N_WINDOW    = 3      # últimos n partidos para el rolling

team_hist = {}
records   = []

for _, row in matches.iterrows():
    home, away = row['home_team'], row['away_team']
    h_hist = team_hist.get(home, [])[-N_WINDOW:]
    a_hist = team_hist.get(away, [])[-N_WINDOW:]

    def avg(hist, key, prior):
        return float(np.mean([x[key] for x in hist])) if hist else prior

    records.append({
        'home_pts_last3': avg(h_hist, 'pts',  PRIOR_PTS),
        'home_gf_last3' : avg(h_hist, 'gf',   PRIOR_GOALS),
        'home_ga_last3' : avg(h_hist, 'ga',   PRIOR_GOALS),
        'away_pts_last3': avg(a_hist, 'pts',  PRIOR_PTS),
        'away_gf_last3' : avg(a_hist, 'gf',   PRIOR_GOALS),
        'away_ga_last3' : avg(a_hist, 'ga',   PRIOR_GOALS),
    })

    ftr  = row['ftr']
    hg, ag = int(row['fthg']), int(row['ftag'])
    h_pts  = 3 if ftr=='H' else (1 if ftr=='D' else 0)
    a_pts  = 3 if ftr=='A' else (1 if ftr=='D' else 0)
    team_hist.setdefault(home, []).append({'pts': h_pts, 'gf': hg, 'ga': ag})
    team_hist.setdefault(away, []).append({'pts': a_pts, 'gf': ag, 'ga': hg})

rolling_df = pd.DataFrame(records, index=matches.index)

print('Features rolling construidas (protocolo anti-leakage):')
print(f'  Window = {N_WINDOW} partidos  |  Prior PTS={PRIOR_PTS}  |  Prior GOALS={PRIOR_GOALS}')
print()
print(f"  {'Feature':<22}  {'μ':>7}  {'σ':>7}  {'min':>7}  {'max':>7}")
print('  ' + '─'*55)
for col in rolling_df.columns:
    s = rolling_df[col]
    print(f'  {col:<22}  {s.mean():>7.3f}  {s.std():>7.3f}  {s.min():>7.2f}  {s.max():>7.2f}')


---
## SECCIÓN 4: Auditoría VIF — Multicolinealidad


In [ ]:
direct_cols  = ['b365h', 'b365a', 'avgd']
rolling_cols = ['home_pts_last3','away_pts_last3',
                'home_gf_last3','home_ga_last3',
                'away_gf_last3','away_ga_last3']
cand_cols    = direct_cols + rolling_cols

df_work = pd.concat([
    matches[direct_cols + ['ftr', 'b365d']],
    rolling_df[rolling_cols]
], axis=1).dropna()
y        = df_work['ftr']
X_cands  = df_work[cand_cols]

def compute_vif(X_df):
    vals = X_df.values.astype(float)
    return pd.Series([variance_inflation_factor(vals, i)
                      for i in range(vals.shape[1])], index=X_df.columns)

def iterative_vif(X_df, threshold=10.0):
    remaining = list(X_df.columns)
    log = []
    while len(remaining) > 1:
        vif_s  = compute_vif(X_df[remaining])
        max_v  = vif_s.max()
        if max_v <= threshold:
            break
        worst = vif_s.idxmax()
        log.append((worst, round(max_v, 2)))
        remaining.remove(worst)
    return remaining, log

print('='*62)
print('AUDITORÍA VIF — ELIMINACIÓN ITERATIVA (umbral VIF ≤ 10)')
print('='*62)
print()
print('VIF inicial:')
vif_init = compute_vif(X_cands)
print(f"  {'Feature':<22}  {'VIF':>8}")
print('  ' + '─'*32)
for feat, v in vif_init.sort_values(ascending=False).items():
    tag = '  ← eliminar' if v > 10 else ''
    print(f'  {feat:<22}  {v:>8.2f}{tag}')
print()

vif_survivors, rm_log = iterative_vif(X_cands)
print('Poda iterativa:')
for feat, v in rm_log:
    print(f'  Eliminada: {feat:<22} (VIF={v})')
print()
print('VIF final (supervivientes):')
vif_final = compute_vif(X_cands[vif_survivors])
print(f"  {'Feature':<22}  {'VIF':>8}  Estado")
print('  ' + '─'*40)
for feat, v in vif_final.sort_values(ascending=False).items():
    print(f'  {feat:<22}  {v:>8.2f}  ✓')
print(f'\n  Resultado: {len(vif_survivors)} features pasan → {vif_survivors}')


---
## SECCIÓN 5: Filtro Spearman — Diagnóstico de Significancia

**Codificación del target:** H=+1 (local gana), D=0 (empate), A=−1 (visitante gana). Umbral α=0.15 — más permisivo que 0.05 para no descartar señales débiles pero reales en n=291.


In [ ]:
y_enc_spr = y.map({'H':1,'D':0,'A':-1})
ALPHA     = 0.15

print('='*76)
print(f'FILTRO SPEARMAN — SIGNIFICANCIA ESTADÍSTICA (α={ALPHA})')
print('='*76)
print(f"  {'Feature':<22}  {'ρ':>8}  {'p-value':>10}  {'Sig.':>5}  Interpretación")
print('  ' + '─'*72)

spearman_res  = {}
final_features = []
rejected       = []

for col in vif_survivors:
    rho, pval = spearmanr(X_cands[col], y_enc_spr)
    spearman_res[col] = {'rho': rho, 'p': pval}
    pasa = pval < ALPHA
    sig  = '✓' if pasa else '✗'
    strength = 'fuerte' if abs(rho)>0.25 else ('moderada' if abs(rho)>0.10 else 'débil')
    direction = '→ más H' if rho>0 else '→ más A'
    interp = f'significativa {strength} {direction}' if pasa else 'no significativa (p>α)'
    print(f'  {col:<22}  {rho:>+8.4f}  {pval:>10.4f}  {sig:>5}  {interp}')
    if pasa:
        final_features.append(col)
    else:
        rejected.append(col)

if len(final_features) < 2:
    final_features = vif_survivors
    print('\n  → Fallback: usando todas las que pasaron VIF')

print(f'\n  Rechazadas (p ≥ {ALPHA}): {rejected}')
print(f'  Features finales ({len(final_features)}): {final_features}')

X = df_work[final_features]


---
## SECCIÓN 6: Entrenamiento — Regresión Logística Multinomial + StratifiedKFold(k=5)

### ¿Por qué `class_weight='balanced'`?

Sin `balanced`, el modelo ignora los empates (solo ~26% del dataset) y jamás predice Draw — un clasificador binario H/A disfrazado. Con `balanced`, el optimizador pondera la pérdida de cada clase por `n_total / (3 × n_clase)`, obligando al modelo a esforzarse en las tres clases por igual.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_tr   = scaler.fit_transform(X_train)
X_te   = scaler.transform(X_test)

n_total = len(y)
print('='*64)
print('CONFIGURACIÓN DEL MODELO')
print('='*64)
print(f'  Algoritmo        : LogisticRegression (multinomial)')
print(f'  Solver           : lbfgs')
print(f'  class_weight     : balanced')
print(f'  Preprocesamiento : StandardScaler')
print(f'  Split            : 80/20 estratificado')
print(f'  Train set        : {len(X_train)} partidos')
print(f'  Test set         : {len(X_test)} partidos')
print(f'  Features finales : {final_features}')
print(f'  Validación       : StratifiedKFold(k=5)')
print()

clf = LogisticRegression(solver='lbfgs', class_weight='balanced',
                          max_iter=1000, random_state=42)

cv        = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(clf, X_tr, y_train, cv=cv, scoring='accuracy')

print('StratifiedKFold CV (5 folds) sobre train set:')
for i, s in enumerate(cv_scores):
    bar = '█' * int(s * 30)
    print(f'  Fold {i+1}: {s*100:5.1f}%  {bar}')
print('  ' + '─'*42)
print(f'  Promedio: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')

clf.fit(X_tr, y_train)
y_pred  = clf.predict(X_te)
y_proba = clf.predict_proba(X_te)
acc     = accuracy_score(y_test, y_pred)

# Versión sin balanced para comparativa
clf_nb = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)
clf_nb.fit(X_tr, y_train)
y_pred_nb = clf_nb.predict(X_te)
acc_nb    = accuracy_score(y_test, y_pred_nb)

print()
print(f'  Test Accuracy (balanced)    : {acc*100:.2f}%')
print(f'  Test Accuracy (sin balanced): {acc_nb*100:.2f}%')


---
## SECCIÓN 7: Resultados — Visualizaciones


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 7A. Accuracy comparativa
ax = axes[0]
test_orig = df_work.loc[X_test.index]
def b365_pred(row):
    odds = {'H': float(row['b365h']), 'D': float(row['b365d']), 'A': float(row['b365a'])}
    return min(odds, key=odds.get)
y_b365   = test_orig.apply(b365_pred, axis=1)
acc_b365 = accuracy_score(y_test, y_b365)

labels_acc = ['Baseline\nguía', 'Bet365\n(test)', 'V1\n(solo cuotas)', 'V2b\n(balanced)★']
accs       = [0.498, acc_b365, 0.5593, acc]
cols_acc   = ['gray', PL_WHITE, PL_PURPLE, PL_CYAN]
bars = ax.bar(labels_acc, [a*100 for a in accs], color=cols_acc, edgecolor=PL_PURPLE, linewidth=1.2)
for bar, val in zip(bars, accs):
    ax.text(bar.get_x()+bar.get_width()/2, val*100+0.3,
            f'{val*100:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.axhline(49.8, color=PL_PINK, linestyle='--', lw=1.5, label='Mínimo exigido')
ax.set_ylim(40, max(accs)*100*1.1)
ax.set_title('Comparativa de Accuracy', color=PL_PURPLE, fontweight='bold')
ax.set_ylabel('Accuracy (%)')
ax.legend(fontsize=9)

# 7B. Matriz de confusión
ax = axes[1]
cm = confusion_matrix(y_test, y_pred, labels=['H','D','A'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Pred H','Pred D','Pred A'],
            yticklabels=['Real H','Real D','Real A'],
            ax=ax, linewidths=0.5)
ax.set_title('Matriz de Confusión (V2b balanced)', color=PL_PURPLE, fontweight='bold')

# 7C. Distribución de confianza
ax = axes[2]
pred_conf    = y_proba.max(axis=1)
correct_mask = (y_pred == y_test.values)
ax.hist(pred_conf[correct_mask],  bins=15, color=PL_CYAN,  alpha=0.7, label='Correctas',  density=True)
ax.hist(pred_conf[~correct_mask], bins=15, color=PL_PINK,  alpha=0.7, label='Incorrectas', density=True)
ax.set_title('Distribución de Confianza\n(P max por predicción)', color=PL_PURPLE, fontweight='bold')
ax.set_xlabel('Probabilidad máxima predicha')
ax.set_ylabel('Densidad')
ax.legend()

plt.suptitle(f'Modelo M3 — LogReg Multinomial | Accuracy={acc*100:.2f}% | CV={cv_scores.mean()*100:.2f}%',
             fontsize=13, fontweight='bold', color=PL_PURPLE, y=1.02)
plt.tight_layout()
plt.savefig('../../img/modelo3_resultados.png', bbox_inches='tight', dpi=150)
plt.show()

print(classification_report(y_test, y_pred, labels=['H','D','A'],
                             target_names=['H (Local)','D (Empate)','A (Visita)'],
                             zero_division=0))


---
## SECCIÓN 8: Comparativa Baselines — Batalla contra Bet365


In [ ]:
draws_bal = int((pd.Series(y_pred)=='D').sum())
draws_nb  = int((pd.Series(y_pred_nb)=='D').sum())

print('='*72)
print('COMPARATIVA COMPLETA DE MODELOS')
print('='*72)
print(f"  {'Predictor':<44}  {'Accuracy':>9}  {'Draws':>6}")
print('  ' + '─'*65)
rows_cmp = [
    ('Baseline guía (mínimo exigido)',              0.498,    '—'),
    ('Bet365 (siempre al favorito, este test set)', acc_b365, '—'),
    ('V1 — solo b365h + b365a (sin balanced)',      0.5593,    '0'),
    ('V2a — rolling+VIF+Spearman (sin balanced)',   acc_nb,   str(draws_nb)),
    ('V2b — rolling+VIF+Spearman (balanced) ★',    acc,      str(draws_bal)),
]
for name, a, d in rows_cmp:
    star = ' ★' if 'V2b' in name else '  '
    print(f'  {name:<44}  {a*100:>8.2f}%  {d:>6}')
print('  ' + '─'*65)
print()
print(f'  Δ V2b vs Bet365      : {(acc-acc_b365)*100:+.2f}pp')
print(f'  Δ V2b vs Baseline    : {(acc-0.498)*100:+.2f}pp')
print()
print('  V1 tiene mayor accuracy pura porque nunca predice empates (sesgo H/A).')
print('  V2b es el único clasificador multiclase honesto — predice las 3 clases.')


---
## SECCIÓN 9: Laboratorio de Feature Engineering Avanzado (ELO + Árbitro)

Para intentar superar el techo de V1, construimos features dinámicas adicionales con protocolo anti-leakage estricto.


In [ ]:
# ELO dinámico y sesgo árbitro — anti-leakage
elo_ratings   = {}
referee_hist  = {}
home_elos, away_elos, ref_bias = [], [], []

for _, row in matches.iterrows():
    home, away, ref = row['home_team'], row['away_team'], row['referee']
    res = row['ftr']

    elo_h = elo_ratings.get(home, 1500.0)
    elo_a = elo_ratings.get(away, 1500.0)
    home_elos.append(elo_h); away_elos.append(elo_a)

    if ref not in referee_hist:
        ref_bias.append(0.44)   # prior: tasa histórica victorias locales PL
    else:
        tot, hw = referee_hist[ref]
        ref_bias.append(hw/tot if tot>0 else 0.44)

    # Actualizar post-partido
    S_h = 1.0 if res=='H' else (0.5 if res=='D' else 0.0)
    E_h = 1 / (1 + 10**((elo_a - elo_h)/400))
    K   = 20
    elo_ratings[home] = elo_h + K*(S_h - E_h)
    elo_ratings[away] = elo_a + K*((1-S_h) - (1-E_h))

    referee_hist.setdefault(ref, [0, 0])
    referee_hist[ref][0] += 1
    if res == 'H': referee_hist[ref][1] += 1

matches['home_elo'] = home_elos
matches['away_elo'] = away_elos
matches['elo_diff'] = matches['home_elo'] - matches['away_elo']
matches['referee_home_bias'] = ref_bias

# Test de las 4 configuraciones
configs = {
    '1. Baseline (solo cuotas)':       ['b365h', 'b365a'],
    '2. Cuotas + Sesgo Árbitro':       ['b365h', 'b365a', 'referee_home_bias'],
    '3. Cuotas + ELO Diff':            ['b365h', 'b365a', 'elo_diff'],
    '4. Súper Modelo (Cuotas+ELO+Ref)':['b365h', 'b365a', 'referee_home_bias', 'elo_diff'],
}

df_lab = matches[['b365h','b365a','b365d','elo_diff','referee_home_bias','ftr']].dropna()
y_lab  = df_lab['ftr']

print('='*65)
print('LABORATORIO — 4 CONFIGURACIONES (sin balanced, para comparar neto)')
print('='*65)
print(f"  {'Configuración':<40}  {'CV Acc':>8}  {'Test Acc':>10}")
print('  ' + '─'*62)

for name, feats in configs.items():
    Xl = df_lab[feats]
    Xtr, Xte, ytr, yte = train_test_split(Xl, y_lab, test_size=0.2,
                                            random_state=42, stratify=y_lab)
    sc  = StandardScaler()
    Xtr_s = sc.fit_transform(Xtr); Xte_s = sc.transform(Xte)
    clf_l = LogisticRegression(solver='lbfgs', random_state=42)
    cv_l  = cross_val_score(clf_l, Xtr_s, ytr, cv=5, scoring='accuracy')
    clf_l.fit(Xtr_s, ytr)
    a_l   = accuracy_score(yte, clf_l.predict(Xte_s))
    print(f'  {name:<40}  {cv_l.mean()*100:>7.2f}%  {a_l*100:>9.2f}%')

print()
print('  Conclusión: añadir ELO y árbitro NO mejora el modelo.')
print('  Las cuotas de Bet365 ya incorporan esa información — añadirla')
print('  genera colinealidad y ruido en un dataset pequeño (n=291).')


---
## SECCIÓN 10: Análisis de Coeficientes — Log-Odds y Odds Ratios

Los coeficientes β están en escala **estandarizada** (StandardScaler). Un incremento de 1σ en la feature produce un cambio de β en el log-odds de esa clase. El **Odds Ratio OR = exp(β)**: OR>1 favorece la clase, OR<1 la perjudica.


In [ ]:
classes = clf.classes_

# Tabla de coeficientes
print('='*80)
print('COEFICIENTES β Y ODDS RATIOS POR CLASE')
print('='*80)
header = f"  {'Feature':<22}"
for cls in classes:
    header += f"    β({cls})   OR({cls})"
print(header)
print('  ' + '─'*76)
for j, feat in enumerate(final_features):
    row_str = f'  {feat:<22}'
    for i, cls in enumerate(classes):
        beta = clf.coef_[i][j]
        OR   = np.exp(beta)
        row_str += f'   {beta:+.4f}  {OR:.4f}'
    print(row_str)

print()
print('Interceptos (probabilidad base sin features):')
for i, cls in enumerate(classes):
    p_base = np.exp(clf.intercept_[i]) / (1 + np.exp(clf.intercept_[i]))
    print(f'  {cls}: β₀={clf.intercept_[i]:+.4f}  P base ≈ {p_base*100:.1f}%')

# Visualización de coeficientes
fig, axes = plt.subplots(1, len(classes), figsize=(5*len(classes), 5))
for i, (cls, ax) in enumerate(zip(classes, axes)):
    betas = pd.Series(clf.coef_[i], index=final_features).sort_values()
    colors_bar = [PL_CYAN if b>0 else PL_PINK for b in betas]
    ax.barh(betas.index, betas.values, color=colors_bar, edgecolor=PL_PURPLE)
    ax.axvline(0, color='white', lw=1.2, linestyle='--')
    ax.set_title(f'Coeficientes β — Clase {cls}', color=PL_PURPLE, fontweight='bold')
    ax.set_xlabel('β estandarizado')

plt.suptitle('Coeficientes por Clase (verde=favorece, rojo=penaliza)',
             fontsize=13, fontweight='bold', color=PL_PURPLE, y=1.02)
plt.tight_layout()
plt.savefig('../../img/modelo3_coeficientes.png', bbox_inches='tight', dpi=150)
plt.show()


---
## RESUMEN EJECUTIVO

| Métrica | Valor |
|---|---|
| **Accuracy Test (V2b balanced)** | ~55% |
| **CV Accuracy (StratifiedKFold k=5)** | ~49% |
| Bet365 accuracy (este test set) | 54.24% |
| Baseline guía (mínimo exigido) | 49.80% |
| Δ vs Bet365 | ~+1pp |
| Δ vs Baseline guía | ~+5pp |
| Features finales | 4 (b365h, b365a, home_pts_last3, away_gf_last3) |
| Candidatos iniciales | 9 |
| Eliminadas por VIF | 3 (avgd, away_pts_last3, home_gf_last3) |
| Eliminadas por Spearman | 2 (home_ga_last3, away_ga_last3) |
| Modelo | LogisticRegression (lbfgs, balanced) |
| Preprocesamiento | StandardScaler |
| Validación | StratifiedKFold(k=5) |
| Empates predichos | >0 (honesto multiclase) |
| V1 sin balanced | 0 empates predichos (clasificador binario disfrazado) |

**Conclusión:** El modelo V2b es el clasificador multiclase honesto — predice las tres clases en serio usando `class_weight='balanced'`. Aunque V1 (solo cuotas, sin balanced) tiene mayor accuracy pura (55.93% vs ~55%), lo logra ignorando completamente los empates, que representan el 26% de los resultados reales. V2b supera el baseline mínimo exigido y a Bet365 en este test set, usando únicamente las cuotas de la propia casa de apuestas más el momentum reciente de los equipos. El laboratorio ELO/árbitro confirma que añadir variables adicionales destruye la precisión — las cuotas de Bet365 ya integran esa información.
